# ============================================================
# Phase 3B - Medical Text (QA) Preprocessing for QGCA
# BTUMQA-225K Standalone Colab Notebook
# ============================================================


# Dual Environment Compatibility Setup & Install Required Libraries


In [1]:
# ── DUAL ENVIRONMENT COMPATIBILITY & DEPENDENCY SETUP ────────────────────────
import os
import sys
from pathlib import Path

def resolve_project_environment(mount_point: str = "/content/drive") -> tuple[Path, Path]:
    try:
        import google.colab
        from google.colab import drive
        drive.mount(mount_point)
        project_root = Path(mount_point) / "MyDrive" / "AUGR-VQA"
        temp_dir = Path("/content")
        print("Running in Google Colab environment.")
    except ImportError:
        # Running locally (parent of notebooks directory)
        project_root = Path(os.getcwd()).parent.resolve()
        temp_dir = project_root / "temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        print("Running in Local environment.")
    return project_root, temp_dir

PROJECT_ROOT, TEMP_DIR = resolve_project_environment()
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q torch transformers pandas tqdm

import json
import math
import time
import statistics
from pathlib import Path
from collections import Counter

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

print("Torch:", torch.__version__)
import transformers
print("Transformers:", transformers.__version__)


Torch: 2.10.0+cu128
Transformers: 5.0.0


# Mount Google Drive and Phase 3B Paths Setup


In [2]:
from pathlib import Path

from google.colab import drive
# drive.mount("/content/drive")

# Change only this project path if needed.
PROJECT_DRIVE_DIR = PROJECT_ROOT

DATASET_RELEASE_NAME = "BTUMQA-225K"
PHASE3A_RELEASE_KEY = "dataset_btumqa_225k"
SPLIT_PREFIX = "btumqa_225k"

PHASE3A_BASE_DIR = PROJECT_DRIVE_DIR / "phase_3" / "p3a_brats_vqa_dataset"
PHASE3B_BASE_DIR = PROJECT_DRIVE_DIR / "phase_3" / "p3b_text_preprocessing"
PHASE3B_BASE_DIR.mkdir(parents=True, exist_ok=True)

PHASE3A_DIR = PHASE3A_BASE_DIR / "dataset_btumqa_225k"
PHASE3B_DIR = PHASE3B_BASE_DIR / "dataset_btumqa_225k"
PHASE3B_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SPLIT_ROWS = {
    "train": 157500,
    "val": 33750,
    "test": 33750,
}

# Input CSVs
TRAIN_CSV_PATH = PHASE3A_DIR / f"{SPLIT_PREFIX}_train.csv"
VAL_CSV_PATH = PHASE3A_DIR / f"{SPLIT_PREFIX}_val.csv"
TEST_CSV_PATH = PHASE3A_DIR / f"{SPLIT_PREFIX}_test.csv"

# Output files
TOKENIZED_TRAIN_PATH = PHASE3B_DIR / "tokenized_qa_train.pt"
TOKENIZED_VAL_PATH = PHASE3B_DIR / "tokenized_qa_val.pt"
TOKENIZED_TEST_PATH = PHASE3B_DIR / "tokenized_qa_test.pt"

EMBEDDINGS_TRAIN_PATH = PHASE3B_DIR / "text_embeddings_train.pt"
EMBEDDINGS_VAL_PATH = PHASE3B_DIR / "text_embeddings_val.pt"
EMBEDDINGS_TEST_PATH = PHASE3B_DIR / "text_embeddings_test.pt"

ANSWER_VOCAB_PATH = PHASE3B_DIR / "answer_vocab.json"
QUESTION_FAMILY_VOCAB_PATH = PHASE3B_DIR / "question_family_vocab.json"
QUESTION_STYLE_VOCAB_PATH = PHASE3B_DIR / "question_style_vocab.json"
DIFFICULTY_LEVEL_VOCAB_PATH = PHASE3B_DIR / "difficulty_level_vocab.json"
AMBIGUITY_FLAG_VOCAB_PATH = PHASE3B_DIR / "ambiguity_flag_vocab.json"
SIGNAL_GAP_BUCKET_VOCAB_PATH = PHASE3B_DIR / "signal_gap_bucket_vocab.json"
REGION_TARGET_VOCAB_PATH = PHASE3B_DIR / "region_target_vocab.json"
DECISION_RULE_VOCAB_PATH = PHASE3B_DIR / "decision_rule_vocab.json"
LABEL_PROVENANCE_VOCAB_PATH = PHASE3B_DIR / "label_provenance_vocab.json"
CANDIDATE_KEEP_REASON_VOCAB_PATH = PHASE3B_DIR / "candidate_keep_reason_vocab.json"

CONFIG_PATH = PHASE3B_DIR / "phase3b_text_config.json"
REPORT_PATH = PHASE3B_DIR / "phase3b_text_preprocessing_report.json"

HF_CACHE_DIR = PHASE3B_DIR / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DRIVE_DIR)
print("Dataset release name:", DATASET_RELEASE_NAME)
print("Phase 3A input dir:", PHASE3A_DIR)
print("Phase 3B output dir:", PHASE3B_DIR)
print("Split prefix:", SPLIT_PREFIX)
print("Expected split rows:", EXPECTED_SPLIT_ROWS)
print()
print("Train CSV exists:", TRAIN_CSV_PATH.exists(), TRAIN_CSV_PATH)
print("Val CSV exists:", VAL_CSV_PATH.exists(), VAL_CSV_PATH)
print("Test CSV exists:", TEST_CSV_PATH.exists(), TEST_CSV_PATH)


Mounted at /content/drive
Project directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging
Dataset release name: BTUMQA-225K
Phase 3A input dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k
Phase 3B output dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k
Split prefix: btumqa_225k
Expected split rows: {'train': 157500, 'val': 33750, 'test': 33750}

Train CSV exists: True /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k/btumqa_225k_train.csv
Val CSV exists: True /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k/btumqa_225k_val.csv


# Phase 3B Configuration
# Resume-Safe Colab Version


In [3]:
ENCODER_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

MAX_LENGTH = 64
BATCH_SIZE = 32
CHUNK_SIZE = 512

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPECTED_ANSWER_VOCAB = [
    "ambiguous",
    "close_gap",
    "confident_present",
    "consistent",
    "context",
    "distinct",
    "edema",
    "enhancing",
    "global",
    "large_confident",
    "large_uncertain",
    "moderate_confident",
    "moderate_gap",
    "moderate_uncertain",
    "ncr_net",
    "none",
    "not_present",
    "shifted",
    "small_confident",
    "small_uncertain",
    "tumor",
    "uncertain_present",
    "wide_gap",
]

EXPECTED_QUESTION_FAMILY_VOCAB = [
    "ambiguous_subregion_pair",
    "confidence_qualified_extent",
    "confidence_qualified_presence",
    "dominant_region_under_uncertainty",
    "highest_risk_region",
    "lowest_risk_region",
    "more_reliable_region",
    "more_uncertain_region",
    "reliability_gap_bucket",
    "safe_region_for_reasoning",
    "uncertainty_consistency_check",
    "uncertainty_gap_bucket",
]

EXPECTED_QUESTION_STYLE_VOCAB = [
    "ambiguity_sensitive",
    "bucketed",
    "comparative",
    "confidence_qualified",
    "ranking",
]

EXPECTED_DIFFICULTY_LEVEL_VOCAB = ["easy", "medium", "hard"]
EXPECTED_AMBIGUITY_FLAG_VOCAB = ["no", "yes"]
EXPECTED_SIGNAL_GAP_BUCKET_VOCAB = ["close_gap", "moderate_gap", "wide_gap"]
EXPECTED_REGION_TARGET_VOCAB = ["", "edema", "ncr_net", "enhancing", "tumor", "context", "global"]

OVERWRITE_EXISTING_TOKENIZED = False
OVERWRITE_EXISTING_TEXT_EMBEDDINGS = False
OVERWRITE_EXISTING_CHUNKS = False

CHUNK_DIR = PHASE3B_DIR / "embedding_chunks"
DONE_DIR = PHASE3B_DIR / "done"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
DONE_DIR.mkdir(parents=True, exist_ok=True)

expected_chunk_counts = {
    split_name: math.ceil(row_count / CHUNK_SIZE)
    for split_name, row_count in EXPECTED_SPLIT_ROWS.items()
}

print("Dataset release name:", DATASET_RELEASE_NAME)
print("Encoder:", ENCODER_NAME)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Chunk size:", CHUNK_SIZE)
print("Device:", DEVICE)
print("Expected answer classes:", len(EXPECTED_ANSWER_VOCAB))
print("Expected question families:", len(EXPECTED_QUESTION_FAMILY_VOCAB))
print("Expected split rows:", EXPECTED_SPLIT_ROWS)
print("Expected chunk counts:", expected_chunk_counts)
print("Chunk directory:", CHUNK_DIR)
print("Done directory:", DONE_DIR)
print("Overwrite tokenized:", OVERWRITE_EXISTING_TOKENIZED)
print("Overwrite final embeddings:", OVERWRITE_EXISTING_TEXT_EMBEDDINGS)
print("Overwrite chunks:", OVERWRITE_EXISTING_CHUNKS)


Dataset release name: BTUMQA-225K
Encoder: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Max length: 64
Batch size: 32
Chunk size: 512
Device: cuda
Expected answer classes: 23
Expected question families: 12
Expected split rows: {'train': 157500, 'val': 33750, 'test': 33750}
Expected chunk counts: {'train': 308, 'val': 66, 'test': 66}
Chunk directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/embedding_chunks
Done directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/done
Overwrite tokenized: False
Overwrite final embeddings: False
Overwrite chunks: False


# Persistence Helpers


In [4]:
def now_string():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def atomic_write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    temp_path.replace(path)


def atomic_torch_save(payload: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")

    torch.save(payload, temp_path)
    temp_path.replace(path)


print("Persistence helpers ready.")


Persistence helpers ready.


# Load Phase 3A QA CSVs


In [5]:
def load_split_csv(path: Path, split_name: str):
    if not path.exists():
        raise FileNotFoundError(f"{split_name} CSV not found: {path}")

    df = pd.read_csv(path, dtype=str)

    required_columns = [
        "qa_id",
        "split",
        "patient_id",
        "slice_id",
        "unique_id",
        "question",
        "answer",
        "question_family",
        "question_style",
        "difficulty_level",
        "ambiguity_flag",
        "region_target_primary",
        "region_target_secondary",
        "signal_gap_bucket",
        "decision_rule_id",
        "label_provenance",
        "candidate_keep_reason",
        "slice_index_in_ugtm_file",
        "phase2c_file",
    ]

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"{split_name} missing required columns: {missing_columns}")

    df = df.fillna("").reset_index(drop=True)

    print(f"{split_name}: rows={len(df)}")
    print("Question families:", dict(df["question_family"].value_counts().sort_index()))
    print("Question styles:", dict(df["question_style"].value_counts().sort_index()))
    print("Difficulty levels:", dict(df["difficulty_level"].value_counts().sort_index()))
    print("Answers:", sorted(df["answer"].unique().tolist()))
    print()

    return df


print("Loading Phase 3A CSVs for:", DATASET_RELEASE_NAME)

df_train = load_split_csv(TRAIN_CSV_PATH, "train")
df_val = load_split_csv(VAL_CSV_PATH, "val")
df_test = load_split_csv(TEST_CSV_PATH, "test")

assert len(df_train) == EXPECTED_SPLIT_ROWS["train"], f"Unexpected train rows: {len(df_train)}"
assert len(df_val) == EXPECTED_SPLIT_ROWS["val"], f"Unexpected val rows: {len(df_val)}"
assert len(df_test) == EXPECTED_SPLIT_ROWS["test"], f"Unexpected test rows: {len(df_test)}"

print("Phase 3A CSVs loaded successfully for:", DATASET_RELEASE_NAME)


Loading Phase 3A CSVs for: BTUMQA-225K
train: rows=157500
Question families: {'ambiguous_subregion_pair': np.int64(15751), 'confidence_qualified_extent': np.int64(11812), 'confidence_qualified_presence': np.int64(11812), 'dominant_region_under_uncertainty': np.int64(15751), 'highest_risk_region': np.int64(11812), 'lowest_risk_region': np.int64(11812), 'more_reliable_region': np.int64(12600), 'more_uncertain_region': np.int64(12600), 'reliability_gap_bucket': np.int64(12600), 'safe_region_for_reasoning': np.int64(12600), 'uncertainty_consistency_check': np.int64(15750), 'uncertainty_gap_bucket': np.int64(12600)}
Question styles: {'ambiguity_sensitive': np.int64(47252), 'bucketed': np.int64(25200), 'comparative': np.int64(25200), 'confidence_qualified': np.int64(23624), 'ranking': np.int64(36224)}
Difficulty levels: {'easy': np.int64(38017), 'hard': np.int64(54750), 'medium': np.int64(64733)}
Answers: ['ambiguous', 'close_gap', 'confident_present', 'consistent', 'context', 'distinct', 'e

# Build Answer and Benchmark-Metadata Vocabularies


In [6]:
def assert_vocab_match(name: str, found_values: list[str], expected_values: list[str]):
    if sorted(found_values) != sorted(expected_values):
        raise ValueError(
            f"{name} mismatch.\n"
            f"Expected(sorted): {sorted(expected_values)}\n"
            f"Found(sorted):    {sorted(found_values)}"
        )


def build_vocab(values: list[str]):
    return {value: idx for idx, value in enumerate(values)}


df_all = pd.concat([df_train, df_val, df_test], axis=0).reset_index(drop=True)

answer_values = sorted(df_all["answer"].unique().tolist())
question_family_values = sorted(df_all["question_family"].unique().tolist())
question_style_values = sorted(df_all["question_style"].unique().tolist())
difficulty_level_values = sorted(df_all["difficulty_level"].unique().tolist())
ambiguity_flag_values = sorted(df_all["ambiguity_flag"].unique().tolist())
signal_gap_bucket_values = sorted(df_all["signal_gap_bucket"].unique().tolist())
region_target_values = sorted(set(df_all["region_target_primary"].tolist() + df_all["region_target_secondary"].tolist()))
decision_rule_values = sorted(df_all["decision_rule_id"].unique().tolist())
label_provenance_values = sorted(df_all["label_provenance"].unique().tolist())
candidate_keep_reason_values = sorted(df_all["candidate_keep_reason"].unique().tolist())

assert_vocab_match("Answer vocabulary", answer_values, EXPECTED_ANSWER_VOCAB)
assert_vocab_match("Question family vocabulary", question_family_values, EXPECTED_QUESTION_FAMILY_VOCAB)
assert_vocab_match("Question style vocabulary", question_style_values, EXPECTED_QUESTION_STYLE_VOCAB)
assert_vocab_match("Difficulty level vocabulary", difficulty_level_values, EXPECTED_DIFFICULTY_LEVEL_VOCAB)
assert_vocab_match("Ambiguity flag vocabulary", ambiguity_flag_values, EXPECTED_AMBIGUITY_FLAG_VOCAB)
assert_vocab_match("Signal gap bucket vocabulary", signal_gap_bucket_values, EXPECTED_SIGNAL_GAP_BUCKET_VOCAB)
assert_vocab_match("Region target vocabulary", region_target_values, EXPECTED_REGION_TARGET_VOCAB)

answer_vocab = build_vocab(EXPECTED_ANSWER_VOCAB)
question_family_vocab = build_vocab(EXPECTED_QUESTION_FAMILY_VOCAB)
question_style_vocab = build_vocab(EXPECTED_QUESTION_STYLE_VOCAB)
difficulty_level_vocab = build_vocab(EXPECTED_DIFFICULTY_LEVEL_VOCAB)
ambiguity_flag_vocab = build_vocab(EXPECTED_AMBIGUITY_FLAG_VOCAB)
signal_gap_bucket_vocab = build_vocab(EXPECTED_SIGNAL_GAP_BUCKET_VOCAB)
region_target_vocab = build_vocab(EXPECTED_REGION_TARGET_VOCAB)
decision_rule_vocab = build_vocab(decision_rule_values)
label_provenance_vocab = build_vocab(label_provenance_values)
candidate_keep_reason_vocab = build_vocab(candidate_keep_reason_values)

print("Answer vocab:")
print(json.dumps(answer_vocab, indent=2))
print("Question family vocab:")
print(json.dumps(question_family_vocab, indent=2))
print("Question style vocab:")
print(json.dumps(question_style_vocab, indent=2))
print("Difficulty level vocab:")
print(json.dumps(difficulty_level_vocab, indent=2))
print("Ambiguity flag vocab:")
print(json.dumps(ambiguity_flag_vocab, indent=2))
print("Signal gap bucket vocab:")
print(json.dumps(signal_gap_bucket_vocab, indent=2))
print("Region target vocab:")
print(json.dumps(region_target_vocab, indent=2))

atomic_write_json(ANSWER_VOCAB_PATH, answer_vocab)
atomic_write_json(QUESTION_FAMILY_VOCAB_PATH, question_family_vocab)
atomic_write_json(QUESTION_STYLE_VOCAB_PATH, question_style_vocab)
atomic_write_json(DIFFICULTY_LEVEL_VOCAB_PATH, difficulty_level_vocab)
atomic_write_json(AMBIGUITY_FLAG_VOCAB_PATH, ambiguity_flag_vocab)
atomic_write_json(SIGNAL_GAP_BUCKET_VOCAB_PATH, signal_gap_bucket_vocab)
atomic_write_json(REGION_TARGET_VOCAB_PATH, region_target_vocab)
atomic_write_json(DECISION_RULE_VOCAB_PATH, decision_rule_vocab)
atomic_write_json(LABEL_PROVENANCE_VOCAB_PATH, label_provenance_vocab)
atomic_write_json(CANDIDATE_KEEP_REASON_VOCAB_PATH, candidate_keep_reason_vocab)

print("Saved vocabularies into:", PHASE3B_DIR)


Answer vocab:
{
  "ambiguous": 0,
  "close_gap": 1,
  "confident_present": 2,
  "consistent": 3,
  "context": 4,
  "distinct": 5,
  "edema": 6,
  "enhancing": 7,
  "global": 8,
  "large_confident": 9,
  "large_uncertain": 10,
  "moderate_confident": 11,
  "moderate_gap": 12,
  "moderate_uncertain": 13,
  "ncr_net": 14,
  "none": 15,
  "not_present": 16,
  "shifted": 17,
  "small_confident": 18,
  "small_uncertain": 19,
  "tumor": 20,
  "uncertain_present": 21,
  "wide_gap": 22
}
Question family vocab:
{
  "ambiguous_subregion_pair": 0,
  "confidence_qualified_extent": 1,
  "confidence_qualified_presence": 2,
  "dominant_region_under_uncertainty": 3,
  "highest_risk_region": 4,
  "lowest_risk_region": 5,
  "more_reliable_region": 6,
  "more_uncertain_region": 7,
  "reliability_gap_bucket": 8,
  "safe_region_for_reasoning": 9,
  "uncertainty_consistency_check": 10,
  "uncertainty_gap_bucket": 11
}
Question style vocab:
{
  "ambiguity_sensitive": 0,
  "bucketed": 1,
  "comparative": 2,
  

# Save Phase 3B Config


In [7]:
phase3b_config = {
    "created_at": now_string(),
    "phase": "Phase 3B - Medical Text Preprocessing for QGCA",
    "dataset_release_name": DATASET_RELEASE_NAME,
    "phase3a_release_key": PHASE3A_RELEASE_KEY,
    "phase3a_dir": str(PHASE3A_DIR),
    "phase3b_dir": str(PHASE3B_DIR),
    "split_prefix": SPLIT_PREFIX,
    "expected_split_rows": EXPECTED_SPLIT_ROWS,
    "encoder_name": ENCODER_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "chunk_size": CHUNK_SIZE,
    "device": str(DEVICE),
    "training_strategy": "single_stage_qgca_with_uncertainty_aware_benchmark_metadata",
    "primary_question_embedding": "cls_last_hidden_state",
    "secondary_question_embedding": "pooler_output",
    "answer_vocab_path": str(ANSWER_VOCAB_PATH),
    "question_family_vocab_path": str(QUESTION_FAMILY_VOCAB_PATH),
    "question_style_vocab_path": str(QUESTION_STYLE_VOCAB_PATH),
    "difficulty_level_vocab_path": str(DIFFICULTY_LEVEL_VOCAB_PATH),
    "ambiguity_flag_vocab_path": str(AMBIGUITY_FLAG_VOCAB_PATH),
    "signal_gap_bucket_vocab_path": str(SIGNAL_GAP_BUCKET_VOCAB_PATH),
    "region_target_vocab_path": str(REGION_TARGET_VOCAB_PATH),
    "decision_rule_vocab_path": str(DECISION_RULE_VOCAB_PATH),
    "label_provenance_vocab_path": str(LABEL_PROVENANCE_VOCAB_PATH),
    "candidate_keep_reason_vocab_path": str(CANDIDATE_KEEP_REASON_VOCAB_PATH),
    "chunk_dir": str(CHUNK_DIR),
    "done_dir": str(DONE_DIR),
    "note": (
        "Phase 3B prepares text-side inputs plus benchmark metadata for BTUMQA-225K. "
        "QGCA training happens in the later modeling phase."
    ),
}

atomic_write_json(CONFIG_PATH, phase3b_config)

print(json.dumps(phase3b_config, indent=2))
print("Config saved:", CONFIG_PATH)


{
  "created_at": "2026-04-27 12:30:30",
  "phase": "Phase 3B - Medical Text Preprocessing for QGCA",
  "dataset_release_name": "BTUMQA-225K",
  "phase3a_release_key": "dataset_btumqa_225k",
  "phase3a_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k",
  "phase3b_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k",
  "split_prefix": "btumqa_225k",
  "expected_split_rows": {
    "train": 157500,
    "val": 33750,
    "test": 33750
  },
  "encoder_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "max_length": 64,
  "batch_size": 32,
  "chunk_size": 512,
  "device": "cuda",
  "training_strategy": "single_stage_qgca_with_uncertainty_aware_benchmark_metadata",
  "primary_question_embedding": "cls_last_hidden_state",
  "secondary_question_embedding": "pooler

# Load BiomedBERT Tokenizer and Encoder


In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    ENCODER_NAME,
    cache_dir=str(HF_CACHE_DIR),
)

text_encoder = AutoModel.from_pretrained(
    ENCODER_NAME,
    cache_dir=str(HF_CACHE_DIR),
)

text_encoder = text_encoder.to(DEVICE)
text_encoder.eval()

for param in text_encoder.parameters():
    param.requires_grad = False

print("Tokenizer loaded:", ENCODER_NAME)
print("Text encoder loaded:", ENCODER_NAME)
print("Hidden size:", text_encoder.config.hidden_size)
print("Frozen parameters:", all(not p.requires_grad for p in text_encoder.parameters()))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer loaded: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Text encoder loaded: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Hidden size: 768
Frozen parameters: True


# Tokenization and Embedding Helper Functions


In [11]:
def safe_torch_load(path: Path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def compute_tokenized_lengths(questions):
    lengths = []
    for question in questions:
        encoded = tokenizer(
            question,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )
        lengths.append(len(encoded["input_ids"]))
    return lengths


def embedding_chunk_dir(split_name: str):
    path = CHUNK_DIR / split_name
    path.mkdir(parents=True, exist_ok=True)
    return path


def embedding_chunk_path(split_name: str, chunk_index: int):
    return embedding_chunk_dir(split_name) / f"chunk_{chunk_index:05d}.pt"


def split_done_path(split_name: str):
    return DONE_DIR / f"{split_name}.json"


def expected_chunk_count(row_count: int):
    return math.ceil(row_count / CHUNK_SIZE)


def validate_chunk_payload(payload: dict, expected_rows: int):
    required_keys = [
        "qa_id",
        "cls_last_hidden_state",
        "pooler_output",
        "answer_id",
        "question_family_id",
        "question_style_id",
        "difficulty_level_id",
        "ambiguity_flag_id",
        "signal_gap_bucket_id",
        "region_target_primary_id",
        "region_target_secondary_id",
        "start_index",
        "end_index",
    ]

    for key in required_keys:
        if key not in payload:
            return False, f"missing_key_{key}"

    if len(payload["qa_id"]) != expected_rows:
        return False, "qa_id_count_mismatch"

    if payload["cls_last_hidden_state"].shape != (expected_rows, 768):
        return False, f"bad_cls_shape_{list(payload['cls_last_hidden_state'].shape)}"

    if payload["pooler_output"].shape != (expected_rows, 768):
        return False, f"bad_pooler_shape_{list(payload['pooler_output'].shape)}"

    id_keys = [
        "answer_id",
        "question_family_id",
        "question_style_id",
        "difficulty_level_id",
        "ambiguity_flag_id",
        "signal_gap_bucket_id",
        "region_target_primary_id",
        "region_target_secondary_id",
    ]

    for key in id_keys:
        if payload[key].shape[0] != expected_rows:
            return False, f"{key}_count_mismatch"

    for tensor_name in ["cls_last_hidden_state", "pooler_output"]:
        tensor = payload[tensor_name]
        if torch.isnan(tensor).any() or torch.isinf(tensor).any():
            return False, f"nan_or_inf_{tensor_name}"

    return True, "ok"


def chunk_already_done(split_name: str, chunk_index: int, expected_rows: int):
    if OVERWRITE_EXISTING_CHUNKS:
        return False

    path = embedding_chunk_path(split_name, chunk_index)
    if not path.exists():
        return False

    try:
        payload = safe_torch_load(path, map_location="cpu")
        valid, reason = validate_chunk_payload(payload, expected_rows)
        if not valid:
            print(f"Existing chunk invalid, will recompute: {path.name} | {reason}")
            return False
        return True
    except Exception as error:
        print(f"Could not read existing chunk, will recompute: {path.name} | {error}")
        return False


def encode_enum_series(df: pd.DataFrame, column: str, vocab: dict[str, int]):
    return torch.tensor([vocab[str(value)] for value in df[column].astype(str).tolist()], dtype=torch.long)


def save_tokenized_split(df: pd.DataFrame, split_name: str, tokenized_path: Path):
    questions = df["question"].astype(str).tolist()
    answers = df["answer"].astype(str).tolist()
    question_families = df["question_family"].astype(str).tolist()
    question_styles = df["question_style"].astype(str).tolist()
    difficulty_levels = df["difficulty_level"].astype(str).tolist()
    ambiguity_flags = df["ambiguity_flag"].astype(str).tolist()
    region_target_primary = df["region_target_primary"].astype(str).tolist()
    region_target_secondary = df["region_target_secondary"].astype(str).tolist()
    signal_gap_buckets = df["signal_gap_bucket"].astype(str).tolist()
    decision_rule_ids = df["decision_rule_id"].astype(str).tolist()
    label_provenance = df["label_provenance"].astype(str).tolist()
    candidate_keep_reason = df["candidate_keep_reason"].astype(str).tolist()

    answer_ids = torch.tensor([answer_vocab[answer] for answer in answers], dtype=torch.long)
    question_family_ids = torch.tensor([question_family_vocab[family] for family in question_families], dtype=torch.long)
    question_style_ids = torch.tensor([question_style_vocab[value] for value in question_styles], dtype=torch.long)
    difficulty_level_ids = torch.tensor([difficulty_level_vocab[value] for value in difficulty_levels], dtype=torch.long)
    ambiguity_flag_ids = torch.tensor([ambiguity_flag_vocab[value] for value in ambiguity_flags], dtype=torch.long)
    signal_gap_bucket_ids = torch.tensor([signal_gap_bucket_vocab[value] for value in signal_gap_buckets], dtype=torch.long)
    region_target_primary_ids = torch.tensor([region_target_vocab[value] for value in region_target_primary], dtype=torch.long)
    region_target_secondary_ids = torch.tensor([region_target_vocab[value] for value in region_target_secondary], dtype=torch.long)
    decision_rule_ids_tensor = torch.tensor([decision_rule_vocab[value] for value in decision_rule_ids], dtype=torch.long)
    label_provenance_ids = torch.tensor([label_provenance_vocab[value] for value in label_provenance], dtype=torch.long)
    candidate_keep_reason_ids = torch.tensor([candidate_keep_reason_vocab[value] for value in candidate_keep_reason], dtype=torch.long)

    slice_index_in_ugtm_file = torch.tensor(df["slice_index_in_ugtm_file"].astype(int).tolist(), dtype=torch.long)

    tokenized_lengths = compute_tokenized_lengths(questions)
    truncation_count = int(sum(length > MAX_LENGTH for length in tokenized_lengths))

    encoded = tokenizer(
        questions,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].long()
    attention_mask = encoded["attention_mask"].long()
    token_type_ids = encoded.get("token_type_ids")
    if token_type_ids is not None:
        token_type_ids = token_type_ids.long()

    tokenized_payload = {
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase3a_release_key": PHASE3A_RELEASE_KEY,
        "split": split_name,
        "encoder_name": ENCODER_NAME,
        "max_length": MAX_LENGTH,
        "qa_id": df["qa_id"].astype(str).tolist(),
        "unique_id": df["unique_id"].astype(str).tolist(),
        "patient_id": df["patient_id"].astype(str).tolist(),
        "slice_id": df["slice_id"].astype(str).tolist(),
        "question": questions,
        "answer": answers,
        "answer_id": answer_ids,
        "question_family": question_families,
        "question_family_id": question_family_ids,
        "question_style": question_styles,
        "question_style_id": question_style_ids,
        "difficulty_level": difficulty_levels,
        "difficulty_level_id": difficulty_level_ids,
        "ambiguity_flag": ambiguity_flags,
        "ambiguity_flag_id": ambiguity_flag_ids,
        "region_target_primary": region_target_primary,
        "region_target_primary_id": region_target_primary_ids,
        "region_target_secondary": region_target_secondary,
        "region_target_secondary_id": region_target_secondary_ids,
        "signal_gap_bucket": signal_gap_buckets,
        "signal_gap_bucket_id": signal_gap_bucket_ids,
        "decision_rule_id": decision_rule_ids,
        "decision_rule_id_tensor": decision_rule_ids_tensor,
        "label_provenance": label_provenance,
        "label_provenance_id": label_provenance_ids,
        "candidate_keep_reason": candidate_keep_reason,
        "candidate_keep_reason_id": candidate_keep_reason_ids,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
        "slice_index_in_ugtm_file": slice_index_in_ugtm_file,
        "phase2c_file": df["phase2c_file"].astype(str).tolist(),
        "tokenized_lengths": torch.tensor(tokenized_lengths, dtype=torch.long),
        "truncation_count": truncation_count,
        "source_phase3a_dir": str(PHASE3A_DIR),
    }

    if tokenized_path.exists() and not OVERWRITE_EXISTING_TOKENIZED:
        print(f"Tokenized file already exists, validating and reusing: {tokenized_path}")
        existing = safe_torch_load(tokenized_path, map_location="cpu")
        if existing["input_ids"].shape != (len(df), MAX_LENGTH):
            raise ValueError(f"Existing tokenized shape mismatch for {split_name}")
        if existing["qa_id"] != tokenized_payload["qa_id"]:
            raise ValueError(f"Existing tokenized qa_id order mismatch for {split_name}")
        return existing

    atomic_torch_save(tokenized_payload, tokenized_path)
    print("Saved tokenized file:", tokenized_path)
    return tokenized_payload


def encode_and_save_chunk(split_name: str, tokenized_payload: dict, chunk_index: int):
    row_count = len(tokenized_payload["qa_id"])
    start = chunk_index * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, row_count)
    expected_rows = end - start

    if chunk_already_done(split_name, chunk_index, expected_rows):
        print(f"{split_name} chunk {chunk_index:05d} already done. Skipping.")
        return

    input_ids = tokenized_payload["input_ids"]
    attention_mask = tokenized_payload["attention_mask"]
    token_type_ids = tokenized_payload["token_type_ids"]

    cls_batches = []
    pooler_batches = []

    with torch.no_grad():
        for batch_start in tqdm(range(start, end, BATCH_SIZE), desc=f"{split_name} chunk {chunk_index:05d}", leave=False):
            batch_end = min(batch_start + BATCH_SIZE, end)
            batch = {
                "input_ids": input_ids[batch_start:batch_end].to(DEVICE),
                "attention_mask": attention_mask[batch_start:batch_end].to(DEVICE),
            }

            if token_type_ids is not None:
                batch["token_type_ids"] = token_type_ids[batch_start:batch_end].to(DEVICE)

            outputs = text_encoder(**batch)
            cls_embedding = outputs.last_hidden_state[:, 0, :].detach().cpu().float()
            if getattr(outputs, "pooler_output", None) is None:
                pooler_embedding = cls_embedding.clone()
            else:
                pooler_embedding = outputs.pooler_output.detach().cpu().float()

            cls_batches.append(cls_embedding)
            pooler_batches.append(pooler_embedding)

            del outputs, cls_embedding, pooler_embedding
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    chunk_payload = {
        "dataset_release_name": DATASET_RELEASE_NAME,
        "split": split_name,
        "chunk_index": chunk_index,
        "start_index": start,
        "end_index": end,
        "qa_id": tokenized_payload["qa_id"][start:end],
        "cls_last_hidden_state": torch.cat(cls_batches, dim=0),
        "pooler_output": torch.cat(pooler_batches, dim=0),
        "answer_id": tokenized_payload["answer_id"][start:end].clone(),
        "question_family_id": tokenized_payload["question_family_id"][start:end].clone(),
        "question_style_id": tokenized_payload["question_style_id"][start:end].clone(),
        "difficulty_level_id": tokenized_payload["difficulty_level_id"][start:end].clone(),
        "ambiguity_flag_id": tokenized_payload["ambiguity_flag_id"][start:end].clone(),
        "signal_gap_bucket_id": tokenized_payload["signal_gap_bucket_id"][start:end].clone(),
        "region_target_primary_id": tokenized_payload["region_target_primary_id"][start:end].clone(),
        "region_target_secondary_id": tokenized_payload["region_target_secondary_id"][start:end].clone(),
    }

    valid, reason = validate_chunk_payload(chunk_payload, expected_rows)
    if not valid:
        raise ValueError(f"Chunk validation failed for {split_name} chunk {chunk_index}: {reason}")

    path = embedding_chunk_path(split_name, chunk_index)
    atomic_torch_save(chunk_payload, path)
    print(f"Saved {split_name} chunk {chunk_index:05d}: rows {start}-{end}")


def merge_embedding_chunks(split_name: str, tokenized_payload: dict, embeddings_path: Path):
    row_count = len(tokenized_payload["qa_id"])
    total_chunks = expected_chunk_count(row_count)
    chunk_payloads = []

    for chunk_index in range(total_chunks):
        path = embedding_chunk_path(split_name, chunk_index)
        if not path.exists():
            raise FileNotFoundError(f"Missing chunk for merge: {path}")

        payload = safe_torch_load(path, map_location="cpu")
        start = chunk_index * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, row_count)
        expected_rows = end - start

        valid, reason = validate_chunk_payload(payload, expected_rows)
        if not valid:
            raise ValueError(f"Invalid chunk during merge: {path} | {reason}")

        expected_qa_ids = tokenized_payload["qa_id"][start:end]
        if payload["qa_id"] != expected_qa_ids:
            raise ValueError(f"QA ID order mismatch in chunk {chunk_index} for {split_name}")

        chunk_payloads.append(payload)

    embeddings_payload = {
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase3a_release_key": PHASE3A_RELEASE_KEY,
        "split": split_name,
        "encoder_name": ENCODER_NAME,
        "max_length": MAX_LENGTH,
        "chunk_size": CHUNK_SIZE,
        "qa_id": tokenized_payload["qa_id"],
        "cls_last_hidden_state": torch.cat([payload["cls_last_hidden_state"] for payload in chunk_payloads], dim=0),
        "pooler_output": torch.cat([payload["pooler_output"] for payload in chunk_payloads], dim=0),
        "answer_id": torch.cat([payload["answer_id"] for payload in chunk_payloads], dim=0),
        "question_family_id": torch.cat([payload["question_family_id"] for payload in chunk_payloads], dim=0),
        "question_style_id": torch.cat([payload["question_style_id"] for payload in chunk_payloads], dim=0),
        "difficulty_level_id": torch.cat([payload["difficulty_level_id"] for payload in chunk_payloads], dim=0),
        "ambiguity_flag_id": torch.cat([payload["ambiguity_flag_id"] for payload in chunk_payloads], dim=0),
        "signal_gap_bucket_id": torch.cat([payload["signal_gap_bucket_id"] for payload in chunk_payloads], dim=0),
        "region_target_primary_id": torch.cat([payload["region_target_primary_id"] for payload in chunk_payloads], dim=0),
        "region_target_secondary_id": torch.cat([payload["region_target_secondary_id"] for payload in chunk_payloads], dim=0),
        "source_phase3a_dir": str(PHASE3A_DIR),
    }

    atomic_torch_save(embeddings_payload, embeddings_path)

    done_payload = {
        "dataset_release_name": DATASET_RELEASE_NAME,
        "split": split_name,
        "status": "complete",
        "rows": row_count,
        "chunk_count": total_chunks,
        "embeddings_path": str(embeddings_path),
        "finished_at": now_string(),
    }

    atomic_write_json(split_done_path(split_name), done_payload)
    print("Merged embedding file saved:", embeddings_path)
    print("Done marker saved:", split_done_path(split_name))
    return embeddings_payload


def final_embedding_already_done(split_name: str, tokenized_payload: dict, embeddings_path: Path):
    if OVERWRITE_EXISTING_TEXT_EMBEDDINGS:
        return False

    done_path = split_done_path(split_name)
    if not embeddings_path.exists() or not done_path.exists():
        return False

    try:
        payload = safe_torch_load(embeddings_path, map_location="cpu")
        row_count = len(tokenized_payload["qa_id"])

        if payload["qa_id"] != tokenized_payload["qa_id"]:
            return False

        if payload["cls_last_hidden_state"].shape != (row_count, 768):
            return False

        if payload["pooler_output"].shape != (row_count, 768):
            return False

        for tensor_name in ["cls_last_hidden_state", "pooler_output"]:
            if torch.isnan(payload[tensor_name]).any() or torch.isinf(payload[tensor_name]).any():
                return False

        print(f"Final embedding file already complete for {split_name}. Skipping encoding.")
        return True

    except Exception as error:
        print(f"Existing final embedding invalid for {split_name}, recomputing. Error: {error}")
        return False


def preprocess_text_split_resume_safe(df: pd.DataFrame, split_name: str, tokenized_path: Path, embeddings_path: Path):
    started_at = time.time()
    tokenized_payload = save_tokenized_split(df, split_name, tokenized_path)

    row_count = len(df)
    total_chunks = expected_chunk_count(row_count)

    if not final_embedding_already_done(split_name, tokenized_payload, embeddings_path):
        for chunk_index in range(total_chunks):
            encode_and_save_chunk(split_name, tokenized_payload, chunk_index)
        embeddings_payload = merge_embedding_chunks(split_name, tokenized_payload, embeddings_path)
    else:
        embeddings_payload = safe_torch_load(embeddings_path, map_location="cpu")

    questions = tokenized_payload["question"]
    tokenized_lengths = tokenized_payload["tokenized_lengths"].tolist()

    split_report = {
        "dataset_release_name": DATASET_RELEASE_NAME,
        "split": split_name,
        "rows": row_count,
        "tokenized_path": str(tokenized_path),
        "embeddings_path": str(embeddings_path),
        "chunk_dir": str(embedding_chunk_dir(split_name)),
        "chunk_count": total_chunks,
        "chunk_size": CHUNK_SIZE,
        "question_word_length": {
            "min": int(min(len(q.split()) for q in questions)),
            "max": int(max(len(q.split()) for q in questions)),
            "mean": float(statistics.mean(len(q.split()) for q in questions)),
        },
        "tokenized_length": {
            "min": int(min(tokenized_lengths)),
            "max": int(max(tokenized_lengths)),
            "mean": float(statistics.mean(tokenized_lengths)),
        },
        "truncation_count": int(tokenized_payload["truncation_count"]),
        "input_ids_shape": list(tokenized_payload["input_ids"].shape),
        "attention_mask_shape": list(tokenized_payload["attention_mask"].shape),
        "token_type_ids_shape": None if tokenized_payload["token_type_ids"] is None else list(tokenized_payload["token_type_ids"].shape),
        "cls_last_hidden_state_shape": list(embeddings_payload["cls_last_hidden_state"].shape),
        "pooler_output_shape": list(embeddings_payload["pooler_output"].shape),
        "answer_distribution": dict(Counter(tokenized_payload["answer"])),
        "question_family_distribution": dict(Counter(tokenized_payload["question_family"])),
        "question_style_distribution": dict(Counter(tokenized_payload["question_style"])),
        "difficulty_level_distribution": dict(Counter(tokenized_payload["difficulty_level"])),
        "ambiguity_flag_distribution": dict(Counter(tokenized_payload["ambiguity_flag"])),
        "bad_tensor_count": 0,
        "bad_tensors": [],
        "elapsed_seconds": round(time.time() - started_at, 2),
    }

    for tensor_name in ["cls_last_hidden_state", "pooler_output"]:
        tensor = embeddings_payload[tensor_name]
        if torch.isnan(tensor).any() or torch.isinf(tensor).any():
            split_report["bad_tensors"].append(tensor_name)

    split_report["bad_tensor_count"] = len(split_report["bad_tensors"])

    print(f"\n{split_name} preprocessing complete.")
    print(json.dumps(split_report, indent=2))
    return split_report


print("Resume-safe helper functions ready.")


Resume-safe helper functions ready.


# Execute Phase 3B Text Preprocessing


In [12]:
run_started = time.time()

train_report = preprocess_text_split_resume_safe(
    df_train,
    "train",
    TOKENIZED_TRAIN_PATH,
    EMBEDDINGS_TRAIN_PATH,
)

val_report = preprocess_text_split_resume_safe(
    df_val,
    "val",
    TOKENIZED_VAL_PATH,
    EMBEDDINGS_VAL_PATH,
)

test_report = preprocess_text_split_resume_safe(
    df_test,
    "test",
    TOKENIZED_TEST_PATH,
    EMBEDDINGS_TEST_PATH,
)

print("\nPhase 3B preprocessing finished for:", DATASET_RELEASE_NAME)
print("Elapsed minutes:", round((time.time() - run_started) / 60, 2))


Saved tokenized file: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/tokenized_qa_train.pt


train chunk 00000:   0%|          | 0/16 [00:00<?, ?it/s]

Saved train chunk 00000: rows 0-512


train chunk 00001:   0%|          | 0/16 [00:00<?, ?it/s]

Saved train chunk 00001: rows 512-1024


...

Saved test chunk 00063: rows 32256-32768


test chunk 00064:   0%|          | 0/16 [00:00<?, ?it/s]

Saved test chunk 00064: rows 32768-33280


test chunk 00065:   0%|          | 0/15 [00:00<?, ?it/s]

Saved test chunk 00065: rows 33280-33750
Merged embedding file saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/text_embeddings_test.pt
Done marker saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/done/test.json

test preprocessing complete.
{
  "dataset_release_name": "BTUMQA-225K",
  "split": "test",
  "rows": 33750,
  "tokenized_path": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/tokenized_qa_test.pt",
  "embeddings_path": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/text_embeddings_test.pt",
  "chunk_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Inte

# Validate Phase 3B Outputs


In [13]:
def validate_saved_split(split_name: str, expected_rows: int, tokenized_path: Path, embeddings_path: Path):
    if not tokenized_path.exists():
        raise FileNotFoundError(f"Missing tokenized file: {tokenized_path}")
    if not embeddings_path.exists():
        raise FileNotFoundError(f"Missing embeddings file: {embeddings_path}")

    done_path = split_done_path(split_name)
    if not done_path.exists():
        raise FileNotFoundError(f"Missing done marker: {done_path}")

    tokenized_payload = safe_torch_load(tokenized_path, map_location="cpu")
    embeddings_payload = safe_torch_load(embeddings_path, map_location="cpu")

    assert len(tokenized_payload["qa_id"]) == expected_rows
    assert len(embeddings_payload["qa_id"]) == expected_rows
    assert tokenized_payload["input_ids"].shape == (expected_rows, MAX_LENGTH)
    assert tokenized_payload["attention_mask"].shape == (expected_rows, MAX_LENGTH)

    if tokenized_payload["token_type_ids"] is not None:
        assert tokenized_payload["token_type_ids"].shape == (expected_rows, MAX_LENGTH)

    assert embeddings_payload["cls_last_hidden_state"].shape == (expected_rows, 768)
    assert embeddings_payload["pooler_output"].shape == (expected_rows, 768)

    tensor_alignment_keys = [
        "answer_id",
        "question_family_id",
        "question_style_id",
        "difficulty_level_id",
        "ambiguity_flag_id",
        "signal_gap_bucket_id",
        "region_target_primary_id",
        "region_target_secondary_id",
    ]

    for key in tensor_alignment_keys:
        assert torch.equal(tokenized_payload[key], embeddings_payload[key]), f"{split_name} alignment failed for {key}"

    assert tokenized_payload["qa_id"] == embeddings_payload["qa_id"]

    for tensor_name in ["cls_last_hidden_state", "pooler_output"]:
        tensor = embeddings_payload[tensor_name]
        assert not torch.isnan(tensor).any(), f"{split_name} {tensor_name} contains NaN"
        assert not torch.isinf(tensor).any(), f"{split_name} {tensor_name} contains Inf"

    total_chunks = expected_chunk_count(expected_rows)
    missing_chunks = []

    for chunk_index in range(total_chunks):
        path = embedding_chunk_path(split_name, chunk_index)
        if not path.exists():
            missing_chunks.append(str(path))
            continue

        start = chunk_index * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, expected_rows)
        expected_chunk_rows = end - start

        chunk_payload = safe_torch_load(path, map_location="cpu")
        valid, reason = validate_chunk_payload(chunk_payload, expected_chunk_rows)
        if not valid:
            raise ValueError(f"Invalid chunk: {path} | {reason}")

    if missing_chunks:
        raise FileNotFoundError(f"Missing chunks for {split_name}: {missing_chunks[:5]}")

    return {
        "dataset_release_name": DATASET_RELEASE_NAME,
        "split": split_name,
        "expected_rows": expected_rows,
        "tokenized_file": str(tokenized_path),
        "embeddings_file": str(embeddings_path),
        "done_marker": str(done_path),
        "chunk_dir": str(embedding_chunk_dir(split_name)),
        "chunk_count": total_chunks,
        "input_ids_shape": list(tokenized_payload["input_ids"].shape),
        "attention_mask_shape": list(tokenized_payload["attention_mask"].shape),
        "cls_last_hidden_state_shape": list(embeddings_payload["cls_last_hidden_state"].shape),
        "pooler_output_shape": list(embeddings_payload["pooler_output"].shape),
        "qa_id_alignment": True,
        "label_alignment": True,
        "benchmark_metadata_alignment": True,
        "nan_or_inf": False,
        "chunks_complete": True,
    }


validation_summary = {
    "train": validate_saved_split("train", EXPECTED_SPLIT_ROWS["train"], TOKENIZED_TRAIN_PATH, EMBEDDINGS_TRAIN_PATH),
    "val": validate_saved_split("val", EXPECTED_SPLIT_ROWS["val"], TOKENIZED_VAL_PATH, EMBEDDINGS_VAL_PATH),
    "test": validate_saved_split("test", EXPECTED_SPLIT_ROWS["test"], TOKENIZED_TEST_PATH, EMBEDDINGS_TEST_PATH),
}

print(json.dumps(validation_summary, indent=2))
print("Resume-safe validation complete for:", DATASET_RELEASE_NAME)


{
  "train": {
    "dataset_release_name": "BTUMQA-225K",
    "split": "train",
    "expected_rows": 157500,
    "tokenized_file": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/tokenized_qa_train.pt",
    "embeddings_file": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/text_embeddings_train.pt",
    "done_marker": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/done/train.json",
    "chunk_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k/embedding_chunks/train",
    "chunk_count": 308,
    "input_ids_shape": [
      157500,
      64
    ],
    "attention_mask_shape": [
   

# Save Final Phase 3B Report


In [14]:
phase3b_report = {
    "finished_at": now_string(),
    "phase": "Phase 3B - Medical Text Preprocessing for QGCA",
    "status": "complete",
    "dataset_release_name": DATASET_RELEASE_NAME,
    "phase3a_release_key": PHASE3A_RELEASE_KEY,
    "phase3a_dir": str(PHASE3A_DIR),
    "phase3b_dir": str(PHASE3B_DIR),
    "split_prefix": SPLIT_PREFIX,
    "expected_split_rows": EXPECTED_SPLIT_ROWS,
    "encoder_name": ENCODER_NAME,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "chunk_size": CHUNK_SIZE,
    "device": str(DEVICE),
    "resume_safe": True,
    "overwrite_existing_tokenized": OVERWRITE_EXISTING_TOKENIZED,
    "overwrite_existing_text_embeddings": OVERWRITE_EXISTING_TEXT_EMBEDDINGS,
    "overwrite_existing_chunks": OVERWRITE_EXISTING_CHUNKS,
    "answer_vocab": answer_vocab,
    "question_family_vocab": question_family_vocab,
    "question_style_vocab": question_style_vocab,
    "difficulty_level_vocab": difficulty_level_vocab,
    "ambiguity_flag_vocab": ambiguity_flag_vocab,
    "signal_gap_bucket_vocab": signal_gap_bucket_vocab,
    "region_target_vocab": region_target_vocab,
    "decision_rule_vocab": decision_rule_vocab,
    "label_provenance_vocab": label_provenance_vocab,
    "candidate_keep_reason_vocab": candidate_keep_reason_vocab,
    "split_reports": {
        "train": train_report,
        "val": val_report,
        "test": test_report,
    },
    "validation_summary": validation_summary,
    "outputs": {
        "tokenized_train": str(TOKENIZED_TRAIN_PATH),
        "tokenized_val": str(TOKENIZED_VAL_PATH),
        "tokenized_test": str(TOKENIZED_TEST_PATH),
        "embeddings_train": str(EMBEDDINGS_TRAIN_PATH),
        "embeddings_val": str(EMBEDDINGS_VAL_PATH),
        "embeddings_test": str(EMBEDDINGS_TEST_PATH),
        "answer_vocab": str(ANSWER_VOCAB_PATH),
        "question_family_vocab": str(QUESTION_FAMILY_VOCAB_PATH),
        "question_style_vocab": str(QUESTION_STYLE_VOCAB_PATH),
        "difficulty_level_vocab": str(DIFFICULTY_LEVEL_VOCAB_PATH),
        "ambiguity_flag_vocab": str(AMBIGUITY_FLAG_VOCAB_PATH),
        "signal_gap_bucket_vocab": str(SIGNAL_GAP_BUCKET_VOCAB_PATH),
        "region_target_vocab": str(REGION_TARGET_VOCAB_PATH),
        "decision_rule_vocab": str(DECISION_RULE_VOCAB_PATH),
        "label_provenance_vocab": str(LABEL_PROVENANCE_VOCAB_PATH),
        "candidate_keep_reason_vocab": str(CANDIDATE_KEEP_REASON_VOCAB_PATH),
        "config": str(CONFIG_PATH),
        "report": str(REPORT_PATH),
        "chunk_dir": str(CHUNK_DIR),
        "done_dir": str(DONE_DIR),
    },
    "next_step": (
        "Phase 4 BTUMQA training using question embeddings as queries, "
        "Phase 2C modulated tokens as keys/values, and optional benchmark metadata integration."
    ),
}

atomic_write_json(REPORT_PATH, phase3b_report)

print(json.dumps(phase3b_report, indent=2))
print("Final Phase 3B report saved:", REPORT_PATH)


{
  "finished_at": "2026-04-27 13:03:56",
  "phase": "Phase 3B - Medical Text Preprocessing for QGCA",
  "status": "complete",
  "dataset_release_name": "BTUMQA-225K",
  "phase3a_release_key": "dataset_btumqa_225k",
  "phase3a_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k",
  "phase3b_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k",
  "split_prefix": "btumqa_225k",
  "expected_split_rows": {
    "train": 157500,
    "val": 33750,
    "test": 33750
  },
  "encoder_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "torch_version": "2.10.0+cu128",
  "transformers_version": "5.0.0",
  "max_length": 64,
  "batch_size": 32,
  "chunk_size": 512,
  "device": "cuda",
  "resume_safe": true,
  "overwrite_existing_tokenized": false,
  "overwrite_existing_tex

# Quick Inspect Saved Artifacts


In [15]:
print("Phase 3B output files:")
for path in sorted(PHASE3B_DIR.glob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path.name:45s} {size_mb:8.2f} MB")

print("\nChunk counts:")
for split_name in ["train", "val", "test"]:
    split_chunk_dir = embedding_chunk_dir(split_name)
    chunk_files = sorted(split_chunk_dir.glob("chunk_*.pt"))
    print(f"{split_name}: {len(chunk_files)} chunks in {split_chunk_dir}")

print("\nDone markers:")
for split_name in ["train", "val", "test"]:
    done_path = split_done_path(split_name)
    print(split_name, "done exists:", done_path.exists(), done_path)

print("\nLoading train embedding sample...")
sample_embeddings = safe_torch_load(EMBEDDINGS_TRAIN_PATH, map_location="cpu")
print("Keys:", sample_embeddings.keys())
print("CLS shape:", sample_embeddings["cls_last_hidden_state"].shape)
print("Pooler shape:", sample_embeddings["pooler_output"].shape)
print("Answer ID shape:", sample_embeddings["answer_id"].shape)
print("Question family ID shape:", sample_embeddings["question_family_id"].shape)
print("Question style ID shape:", sample_embeddings["question_style_id"].shape)
print("Difficulty level ID shape:", sample_embeddings["difficulty_level_id"].shape)
print("Ambiguity flag ID shape:", sample_embeddings["ambiguity_flag_id"].shape)
print("Signal gap bucket ID shape:", sample_embeddings["signal_gap_bucket_id"].shape)
print("\nSample QA IDs:", sample_embeddings["qa_id"][:5])


Phase 3B output files:
ambiguity_flag_vocab.json                         0.00 MB
answer_vocab.json                                 0.00 MB
candidate_keep_reason_vocab.json                  0.00 MB
decision_rule_vocab.json                          0.00 MB
difficulty_level_vocab.json                       0.00 MB
label_provenance_vocab.json                       0.00 MB
phase3b_text_config.json                          0.00 MB
phase3b_text_preprocessing_report.json            0.02 MB
question_family_vocab.json                        0.00 MB
question_style_vocab.json                         0.00 MB
region_target_vocab.json                          0.00 MB
signal_gap_bucket_vocab.json                      0.00 MB
text_embeddings_test.pt                         200.94 MB
text_embeddings_train.pt                        937.73 MB
text_embeddings_val.pt                          200.94 MB
tokenized_qa_test.pt                             56.72 MB
tokenized_qa_train.pt                           2